<a href="https://colab.research.google.com/github/OssamaSijbesma/uu-data-analysis-and-machine-learning-assignment-1/blob/main/report.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[Assignment 1: Supervised learning competition](https://daml.robertab.nl/assignments/assignment_1.html)

In [ ]:
# Install packages in notebook
%pip install pyreadr optuna --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.4/788.4 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 19.2 MB/s eta 0:00:00


In [ ]:
import pyreadr
import requests
import io
import optuna
import pandas as pd
import xgboost as xgb
import numpy as np
from sklearn.model_selection import KFold, cross_val_score, RandomizedSearchCV
from sklearn.feature_selection import RFECV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from scipy.stats import randint, uniform, loguniform
import matplotlib.pyplot as plt

GOOGLE_COLAB = True # When using Google Colab set to True.
DATA_SOURCE = "https://github.com/OssamaSijbesma/uu-data-analysis-and-machine-learning-assignment-1/raw/refs/heads/main/"

if(GOOGLE_COLAB):
  # Fetch files from online data source
  train_response = requests.get(DATA_SOURCE+"raw/train.rds")
  test_response = requests.get(DATA_SOURCE+"raw/test.rds")
  train_rds = pyreadr.read_r(io.BytesIO(train_response.content))
  test_rds = pyreadr.read_r(io.BytesIO(test_response.content))
else:
  # Fetch files from local data source
  train_rds = pyreadr.read_r("./raw/train.rds")
  test_rds = pyreadr.read_r("./raw/test.rds")

# Create pandas data frames from files
train_df = pd.DataFrame.from_dict(train_rds[None])
test_df = pd.DataFrame.from_dict(test_rds[None])


# Data description

Describe the data and use a visualization to support your story. (approx. one or two paragraphs)


In [ ]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 316 entries, 0 to 315
Data columns (total 31 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   school      316 non-null    category
 1   sex         316 non-null    category
 2   age         316 non-null    float64 
 3   address     316 non-null    category
 4   famsize     316 non-null    category
 5   Pstatus     316 non-null    category
 6   Medu        316 non-null    float64 
 7   Fedu        316 non-null    float64 
 8   Mjob        316 non-null    category
 9   Fjob        316 non-null    category
 10  reason      316 non-null    category
 11  guardian    316 non-null    category
 12  traveltime  316 non-null    float64 
 13  studytime   316 non-null    float64 
 14  failures    316 non-null    float64 
 15  schoolsup   316 non-null    category
 16  famsup      316 non-null    category
 17  paid        316 non-null    category
 18  activities  316 non-null    category
 19  nursery 

In [ ]:
train_df.head()

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,internet,romantic,famrel,freetime,goout,Dalc,Walc,health,absences,score
0,GP,F,18.0,U,GT3,T,3.0,2.0,other,services,...,yes,yes,5.0,4.0,3.0,2.0,3.0,1.0,7.0,0.648402
1,GP,F,17.0,U,GT3,T,3.0,4.0,services,other,...,yes,no,4.0,4.0,5.0,1.0,3.0,5.0,16.0,1.169019
2,GP,M,17.0,U,GT3,T,2.0,3.0,other,other,...,yes,no,5.0,2.0,2.0,1.0,1.0,2.0,4.0,0.384039
3,GP,M,18.0,R,GT3,T,4.0,3.0,teacher,services,...,yes,yes,5.0,3.0,2.0,1.0,2.0,4.0,9.0,1.207493
4,GP,F,16.0,U,GT3,T,1.0,3.0,at_home,services,...,yes,yes,4.0,3.0,5.0,1.0,1.0,3.0,0.0,-1.215206


In [ ]:
train_df.describe()

,age,Medu,Fedu,traveltime,studytime,failures,famrel,freetime,goout,Dalc,Walc,health,absences,score
count,316.000000,316.000000,316.000000,316.000000,316.000000,316.000000,316.000000,316.000000,316.000000,316.000000,316.000000,316.000000,316.000000,316.000000
mean,16.753165,2.737342,2.503165,1.452532,2.034810,0.370253,3.933544,3.218354,3.066456,1.468354,2.287975,3.465190,5.718354,-0.016197
std,1.302923,1.105828,1.102662,0.690728,0.840667,0.788167,0.904318,0.972533,1.097767,0.881347,1.278385,1.385431,8.303744,0.986209
min,15.000000,0.000000,0.000000,1.000000,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,-2.705322
25%,16.000000,2.000000,2.000000,1.000000,1.000000,0.000000,4.000000,3.000000,2.000000,1.000000,1.000000,3.000000,0.000000,-0.634936
50%,17.000000,3.000000,2.000000,1.000000,2.000000,0.000000,4.000000,3.000000,3.000000,1.000000,2.000000,3.500000,4.000000,-0.027688
75%,18.000000,4.000000,3.000000,2.000000,2.000000,0.000000,5.000000,4.000000,4.000000,2.000000,3.000000,5.000000,8.000000,0.640293
max,22.000000,4.000000,4.000000,4.000000,4.000000,3.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,75.000000,2.234578


In [ ]:
for col in train_df.select_dtypes(include=['object', 'category']).columns:
  print(f"{col}: {train_df[col].unique().tolist()}")

school: ['GP', 'MS']
sex: ['F', 'M']
address: ['U', 'R']
famsize: ['GT3', 'LE3']
Pstatus: ['T', 'A']
Mjob: ['other', 'services', 'teacher', 'at_home', 'health']
Fjob: ['services', 'other', 'at_home', 'teacher', 'health']
reason: ['other', 'course', 'home', 'reputation']
guardian: ['mother', 'father', 'other']
schoolsup: ['no', 'yes']
famsup: ['no', 'yes']
paid: ['no', 'yes']
activities: ['no', 'yes']
nursery: ['yes', 'no']
higher: ['yes', 'no']
internet: ['yes', 'no']
romantic: ['yes', 'no']


# Model description

Briefly describe which models you compare to perform prediction. (approx. two or three paragraphs)

K-NN
Random Forrest
XG-Boost
Log odds

# Data transformation and pre-processing

Describe additional pre-processing steps you have used, if any (e.g., dealing with categorical data, scaling). If you do not do any pre-processing, you can leave this section out.

In [ ]:
y = train_df['score']
X = train_df.drop(columns=['score'])

# convert category or object values into dummies
X_dummy = pd.get_dummies(X, drop_first=True)

# Model comparison

Describe how you compare the methods and why. (approx. two or three paragraphs)

In [ ]:
xbg_model = xgb.XGBRegressor(
    objective='reg:squarederror'
)

param_distributions = {
    'max_depth': randint(2, 30),                   # Integers: 2, 3, 4, 5
    'n_estimators': randint(50, 250),             # Integers: 50 to 249
    'learning_rate': loguniform(0.01, 0.2),       # Log-scaled floats
    'subsample': uniform(0.6, 0.35),              # Uniform floats: [0.60, 0.95]
    'colsample_bytree': uniform(0.6, 0.35),       # Uniform floats: [0.60, 0.95]
    'reg_alpha': loguniform(1e-3, 5.0),           # Log-scaled float
    'reg_lambda': loguniform(1e-2, 10.0)          # Log-scaled float
}

search = RandomizedSearchCV(
    estimator=xbg_model,
    param_distributions=param_distributions,
    n_iter=500, # How many combinations
    scoring='neg_root_mean_squared_error',
    cv=KFold(n_splits=5, shuffle=True), # Cross validation
    n_jobs=-1 # Run parallel using all processors.
)

search.fit(X, y)

print(f"Best CV RMSE: {-search.best_score_:.4f}")
print(f"Best Parameters: {search.best_params_}")

Best CV RMSE: 0.8626
Best Parameters: {'colsample_bytree': np.float64(0.9139966990970666), 'learning_rate': np.float64(0.03507455893172157), 'max_depth': 3, 'n_estimators': 76, 'reg_alpha': np.float64(0.06995999352186495), 'reg_lambda': np.float64(0.03279027091922544), 'subsample': np.float64(0.7017378518166948)}


In [ ]:
def objective(trial):
    params = {
        'max_depth': trial.suggest_int('max_depth', 4, 10),
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0)
    }

    xgb_model = xgb.XGBRegressor(
        objective= 'reg:squarederror',
        n_jobs= -1,
        **params
    )

    # Evaluate across folds using Negative Root Mean Squared Error
    scores = cross_val_score(
        xgb_model, X, y,
        cv=KFold(n_splits=5, shuffle=True, random_state=42),
        scoring='neg_root_mean_squared_error',
        n_jobs=-1
    )

    # Return average CV RMSE (Optuna minimizes this value)
    return np.mean(-scores)

# 3. Create Study & Optimize
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=500, show_progress_bar=True)

# 4. Output Results
print(f"Best CV RMSE: {study.best_value:.4f}")
print("Best Hyperparameters:")
for key, value in study.best_params.items():
    print(f"  {key}={value},")

[I 2026-09-24 09:25:01,756] A new study created in memory with name: no-name-c5d5b5be-92dd-4dd7-ba1a-6724dddeafd7


  0%|          | 0/500 [00:00<?, ?it/s]

[I 2026-09-24 09:25:04,542] Trial 0 finished with value: 0.8864474879822668 and parameters: {'max_depth': 5, 'n_estimators': 167, 'learning_rate': 0.032464748670697874, 'reg_alpha': 8.448592705784595, 'reg_lambda': 0.005731460348820518, 'subsample': 0.939789121552483, 'colsample_bytree': 0.6630522068820542}. Best is trial 0 with value: 0.8864474879822668.
[I 2026-09-24 09:25:05,327] Trial 1 finished with value: 0.8988690066087306 and parameters: {'max_depth': 8, 'n_estimators': 188, 'learning_rate': 0.0055462940190176115, 'reg_alpha': 0.0022412683196441404, 'reg_lambda': 1.315861625955499, 'subsample': 0.7769822212209396, 'colsample_bytree': 0.7317668812466491}. Best is trial 0 with value: 0.8864474879822668.
[I 2026-09-24 09:25:05,571] Trial 2 finished with value: 0.8838915502475784 and parameters: {'max_depth': 7, 'n_estimators': 56, 'learning_rate': 0.024566305007575305, 'reg_alpha': 0.22166135028347686, 'reg_lambda': 0.01616446239996158, 'subsample': 0.5197274330947861, 'colsample_

Random Forest Regression

In [ ]:
# fit the estimator
rf_model = RandomForestRegressor(random_state = 42)
rf_model.fit(X_dummy, y)

# 5-fold cross-validation before hyperparameter tuning
rf_cross_val_scores = cross_val_score(
    rf_model,
    X_dummy,
    y,
    cv=5,
    scoring="neg_mean_squared_error"
)

rf_MSE = -(rf_cross_val_scores)
print(rf_MSE)

[0.81738261 0.64921965 0.74710535 0.86050946 0.86272463]


In [ ]:
# hyperparameter tuning
# check current parameters
rf_model.get_params()

param_distributions = {
    'n_estimators': list(range(50, 301, 50)),   # number of trees
    'max_features': ['sqrt', 'log2', 1.0],      # max number of features considered for splitting a node
    'max_depth': [None, 3, 5, 10, 15],          # max number of levels in each tree
    'min_samples_split': [2, 5, 10],            # min number of data points placed in a node before the node is split
    'min_samples_leaf': [1, 2, 4],              # min number of data points allowed in a leaf node
    'bootstrap': [True, False]                  # sampling data points with or without replacement
}

rf_search = RandomizedSearchCV(
    estimator = rf_model,
    param_distributions = param_distributions,
    n_iter = 100,
    cv = 5,
    scoring="neg_mean_squared_error",
    n_jobs = -1
)

rf_search.fit(X_dummy, y)
rf_search.best_params_

{'n_estimators': 150,
 'min_samples_split': 10,
 'min_samples_leaf': 4,
 'max_features': 1.0,
 'max_depth': 5,
 'bootstrap': True}

In [ ]:
# fit model again with the new hyperparameters
rf_model_best_params = RandomForestRegressor (random_state = 42,
                                              n_estimators = 150,
                                              min_samples_split =10,
                                              min_samples_leaf = 4,
                                              max_features = 1.0,
                                              max_depth = 5,
                                              bootstrap = True)

# 5-fold cross-validation after hyperparameter tuning
rf_cross_val_scores_best_params = cross_val_score(
    rf_model_best_params,
    X_dummy,
    y,
    cv=5,
    scoring="neg_mean_squared_error"
)

rf_MSE_best_params = -(rf_cross_val_scores_best_params)
print(rf_MSE_best_params) # MSE scores seem to be a bit lower

[0.7797422  0.66489943 0.70728408 0.7609683  0.81044186]


# Chosen model

Show which method is best and why. (approx. one paragraph) You are welcome to use tables and plots!

[link text](https://)

# Team member contributions

Write down what each team member contributed to the project.

- Author One: a, b, c
- Author Two: b, c, d
- Author Three: a, b, d